# Install dependencies (Colab)

In [ ]:
import sys

# Remove torchaudio (not used in your project)
!{sys.executable} -m pip uninstall -y torchaudio

# Reinstall your stack
!{sys.executable} -m pip install -U monai[all] torchio tqdm #nibabel scikit-learn

# Make sure filelock is new enough and cannot be downgraded
!{sys.executable} -m pip install -U "filelock>=3.15" --no-deps


Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 39.1 MB/s eta 0:00:00


In [ ]:
# If running in Colab, uncomment this to install deps:
#!pip -q install monai[all] torchio tqdm

import os, gc, sys, json, math, random, time, glob, shutil, textwrap
from datetime import datetime
import numpy as np
import pandas as pd
import nibabel as nib

import torch
torch.backends.cudnn.benchmark = True
import torchio as tio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from monai.metrics import DiceMetric, HausdorffDistanceMetric, MeanIoU
from monai.transforms import Compose, Activations, AsDiscrete

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

##Mount Google Drive & set device

In [ ]:
# Mount Drive on Colab and pick GPU
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not in Colab environment.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
device = "cuda" if torch.cuda.is_available() else "cpu"

Mounted at /content/drive
Device: cuda


# Global config (edit me!)

In [ ]:
# ===== PROJECT ROOT (where you want checkpoints/logs/figures) =====
PROJECT_DIR = "/content/drive/MyDrive/Brain_Tumor_Segmentation"

# ===== DATA ROOT (folder that directly contains the BraTS subject folders) =====
# e.g., ".../ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
DATA_DIR = os.path.join(
    PROJECT_DIR,
    "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)

# ===== EXPERIMENT NAMING =====
EXPERIMENT_NAME = "DHW_baseline"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, f"{EXPERIMENT_NAME}_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

METRICS_CSV = os.path.join(CHECKPOINT_DIR, "epoch_metrics.csv")
LOG_DIR = os.path.join(CHECKPOINT_DIR, "tb_logs")
os.makedirs(LOG_DIR, exist_ok=True)

# ===== SPLIT / TRAIN SETTINGS =====
TRAIN_VAL_TEST_SPLIT = (0.70, 0.15, 0.15)
RANDOM_STATE = 42

NUM_EPOCHS = 50
BATCH_SIZE = 2
NUM_WORKERS = 8
PIN_MEMORY = True
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
AMP = True
GRAD_CLIP_NORM = 0.0

# Resume controls
RESUME_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "epoch_042.pth")   # "" or a full path to a specific epoch .pth
RESUME_EPOCH = 42      # None or an int, e.g., 16

# Logging / saving
SAVE_EVERY_EPOCH = True
PRINT_EVERY = 1

# --- Aug knobs (kept modest for full-volume training) ---
AUG_FLIP_PROB = 0.5

# Affine: keep depth axis stable (we already pool only spatially), tiny rotations & scale
AFFINE_SCALES = (0.95, 1.05)      # narrower than before to avoid label tearing
AFFINE_DEGREES = (5, 5, 5)        # x,y,z degrees (or a single int); small but helpful
AFFINE_TRANSLATION = 4            # voxels (x,y,z each)

# Intensity/contrast
GAMMA_PROB = 0.3
GAMMA_RANGE = (0.9, 1.1)          # mild, keeps anatomy plausible

# Elastic deformation (small): helps edema borders; keep very conservative
ELASTIC_PROB = 0.15
ELASTIC_CTRL_PTS = 5              # control points per dim
ELASTIC_MAX_DISP = 3.0            # voxels

# Motion/blur (very light; helps generalization, don’t overdo)
MOTION_PROB = 0.10
BLUR_PROB   = 0.05
BLUR_STD    = (0.5, 1.0)

# Noise/Bias (you already had these; keep modest)
NOISE_PROB  = 0.5
NOISE_STD   = (0.0, 0.08)         # a tad lower than 0.1 to protect subtle ET
BIAS_FIELD_PROB = 0.25            # slight reduction

# Orientation control (your earlier switch)
DEPTH_FROM = "D"  # "D"→155x240x240, "H"→240x240x155, "W"→240x155x240
assert DEPTH_FROM in {"H","W","D"}

SEED = 1337
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

print("PROJECT_DIR     :", PROJECT_DIR)
print("DATA_DIR        :", DATA_DIR)
print("CHECKPOINT_DIR  :", CHECKPOINT_DIR)
print("METRICS_CSV     :", METRICS_CSV)
print("LOG_DIR         :", LOG_DIR)

PROJECT_DIR     : /content/drive/MyDrive/Brain_Tumor_Segmentation
DATA_DIR        : /content/drive/MyDrive/Brain_Tumor_Segmentation/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData
CHECKPOINT_DIR  : /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints
METRICS_CSV     : /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/epoch_metrics.csv
LOG_DIR         : /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/tb_logs


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# List subjects and create consistent train/val/test splits

In [ ]:
def list_subject_folders(root_dir):
    # Return sorted list of BraTS subject folders (start with 'BraTS-').
    return sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d)) and d.startswith("BraTS-")
    ])

def ensure_splits(root_dir, checkpoint_dir, train_ratio, val_ratio, test_ratio, random_state=42):
    # Create or load persistent subject splits (JSON) for reproducibility.
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Splits must sum to 1.0"
    splits_path = os.path.join(checkpoint_dir, "splits.json")
    all_folders = list_subject_folders(root_dir)

    if os.path.exists(splits_path):
        with open(splits_path, "r") as f:
            sp = json.load(f)
        known = set(sp["train"] + sp["val"] + sp["test"])
        current = set(all_folders)
        if known <= current:
            print("Loaded existing splits:", splits_path)
            return sp["train"], sp["val"], sp["test"]
        else:
            print("Split file found but subjects differ; regenerating splits...")

    from sklearn.model_selection import train_test_split
    train_folders, temp = train_test_split(
        all_folders, train_size=train_ratio, random_state=random_state, shuffle=True
    )
    rel = test_ratio / (val_ratio + test_ratio)
    val_folders, test_folders = train_test_split(
        temp, test_size=rel, random_state=random_state, shuffle=True
    )
    sp = {"train": train_folders, "val": val_folders, "test": test_folders}
    with open(splits_path, "w") as f:
        json.dump(sp, f, indent=2)
    print("Wrote splits to:", splits_path)
    return train_folders, val_folders, test_folders

train_folders, val_folders, test_folders = ensure_splits(
    DATA_DIR, CHECKPOINT_DIR, *TRAIN_VAL_TEST_SPLIT, random_state=RANDOM_STATE
)
print(f"#subjects — train: {len(train_folders)} | val: {len(val_folders)} | test: {len(test_folders)}")


Loaded existing splits: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/splits.json
#subjects — train: 875 | val: 188 | test: 188


# Dataset + DataLoaders (permute H×W×D → D×H×W, TorchIO aug)

In [ ]:
def zscore_nonzero(x: np.ndarray) -> np.ndarray:
    mask = x > 0
    if mask.sum() > 0:
        m = x[mask].mean()
        s = x[mask].std()
        if s > 0:
            x = x.copy()
            x[mask] = (x[mask] - m) / s
        else:
            x = (x - x.mean()) / (x.std() + 1e-8)
    else:
        x = (x - x.mean()) / (x.std() + 1e-8)
    return x.astype(np.float32)

def load_modality_tensor(path: str) -> torch.Tensor:
    # Load a single modality, normalize, permute HWD->DHW, return torch tensor (D,H,W).
    vol = nib.load(path).get_fdata().astype(np.float32)
    vol = zscore_nonzero(vol)
    vol = np.transpose(vol, (2, 0, 1))   # H,W,D -> D,H,W
    return torch.from_numpy(vol)         # (D,H,W)

def build_target_channels(seg_path: str) -> torch.Tensor:
    # Build 3 binary channels (WT, TC, ET) from BraTS labels: 0 bg, 1 NCR/NET, 2 ED, 4 ET.
    seg = nib.load(seg_path).get_fdata().astype(np.int16)   # (H,W,D)
    seg = np.transpose(seg, (2, 0, 1))                      # (D,H,W)

    wt = (seg > 0).astype(np.float32)  # Whole tumor
    tc = ((seg == 1) | (seg == 3)).astype(np.float32)  # Tumor core
    et = (seg == 3).astype(np.float32)                 # Enhancing tumor

    target = np.stack([wt, tc, et], axis=0)  # (3,D,H,W) order: WT, TC, ET
    return torch.from_numpy(target)

class BraTS2023Dataset(Dataset):
    def __init__(self, root_dir, folders, transform=None):
        self.root = root_dir
        self.folders = list(folders)
        self.transform = transform

    def __len__(self):
        return len(self.folders)

    def _paths(self, folder):
        base = os.path.join(self.root, folder, folder)
        seg = base + "-seg.nii.gz"
        t1c = base + "-t1c.nii.gz"
        t1n = base + "-t1n.nii.gz"
        t2f = base + "-t2f.nii.gz"
        t2w = base + "-t2w.nii.gz"
        return t1c, t1n, t2f, t2w, seg

    def __getitem__(self, idx):
        folder = self.folders[idx]
        t1c_path, t1n_path, t2f_path, t2w_path, seg_path = self._paths(folder)

        # Load modalities (D,H,W) and stack to (4,D,H,W)
        t1c = load_modality_tensor(t1c_path)
        t1n = load_modality_tensor(t1n_path)
        t2f = load_modality_tensor(t2f_path)
        t2w = load_modality_tensor(t2w_path)
        image = torch.stack([t1c, t1n, t2f, t2w], dim=0)  # (4,D,H,W)

        # Build target mask (3,D,H,W)
        target = build_target_channels(seg_path)

        # TorchIO requires (C,X,Y,Z). Identity affine is fine.
        subject = tio.Subject(
            image=tio.ScalarImage(tensor=image, affine=np.eye(4)),
            mask=tio.LabelMap(tensor=target, affine=np.eye(4)),
            name=folder
        )

        if self.transform is not None:
            subject = self.transform(subject)

        image = subject.image.data.float()  # (4,D,H,W)
        target = subject.mask.data.float()  # (3,D,H,W)
        return image, target, folder

# TorchIO augmentations for train; identity for val/test
"""
train_transform = tio.Compose([
    # Flips in any axis
    tio.RandomFlip(axes=(0, 1, 2), flip_probability=AUG_FLIP_PROB),

    # Small affine (linear on images, nearest on labels handled by LabelMap)
    tio.RandomAffine(
        scales=AFFINE_SCALES,
        degrees=AFFINE_DEGREES,           # can be int or (x,y,z)
        translation=AFFINE_TRANSLATION,   # can be int or (x,y,z)
        isotropic=False,
        image_interpolation="linear",
        p=0.9
    ),

    # Very light elastic deformation (nearest for LabelMap is automatic)
    tio.RandomElasticDeformation(
        num_control_points=ELASTIC_CTRL_PTS,
        max_displacement=ELASTIC_MAX_DISP,
        p=ELASTIC_PROB
    ),

    # Intensity tweaks: gamma, motion, blur (low probs)
    tio.RandomGamma(log_gamma=GAMMA_RANGE, p=GAMMA_PROB),

    # One of motion blur or gaussian blur (low impact)
    tio.OneOf({
        tio.RandomMotion(): MOTION_PROB,
        tio.RandomBlur(std=BLUR_STD): BLUR_PROB,
    }, p=MOTION_PROB + BLUR_PROB),

    # Noise & bias field
    tio.RandomNoise(mean=0.0, std=NOISE_STD, p=NOISE_PROB),
    tio.RandomBiasField(p=BIAS_FIELD_PROB),
])
"""
train_transform = tio.Compose([])
val_transform = tio.Compose([])
test_transform = tio.Compose([])

train_ds = BraTS2023Dataset(DATA_DIR, train_folders, transform=train_transform)
val_ds   = BraTS2023Dataset(DATA_DIR, val_folders,   transform=val_transform)
test_ds  = BraTS2023Dataset(DATA_DIR, test_folders,  transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("Dataloaders ready.")


Dataloaders ready.


# Model — 3D U-Net

In [ ]:
class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, base_ch=64):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv3D(in_channels, base_ch)         # 4 -> 64
        self.pool1 = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.enc2 = DoubleConv3D(base_ch, base_ch*2)           # 64 -> 128
        self.pool2 = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.enc3 = DoubleConv3D(base_ch*2, base_ch*4)         # 128 -> 256
        self.pool3 = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))

        # Bottleneck
        self.bottleneck = DoubleConv3D(base_ch*4, base_ch*8)   # 256 -> 512

        # Decoder
        self.up3 = nn.ConvTranspose3d(base_ch*8, base_ch*4, kernel_size=(1,2,2), stride=(1,2,2))
        self.dec3 = DoubleConv3D(base_ch*8, base_ch*4)         # concat 256 + 256 -> 256
        self.up2 = nn.ConvTranspose3d(base_ch*4, base_ch*2, kernel_size=(1,2,2), stride=(1,2,2))
        self.dec2 = DoubleConv3D(base_ch*4, base_ch*2)         # concat 128 + 128 -> 128
        self.up1 = nn.ConvTranspose3d(base_ch*2, base_ch,     kernel_size=(1,2,2), stride=(1,2,2))
        self.dec1 = DoubleConv3D(base_ch*2, base_ch)           # concat 64 + 64 -> 64

        # Output
        self.out_conv = nn.Conv3d(base_ch, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        # Bottleneck
        b = self.bottleneck(p3)

        # Decoder
        u3 = self.up3(b)
        if u3.shape[2:] != e3.shape[2:]:
            u3 = self._center_crop_or_pad(u3, e3.shape[2:])
        d3 = self.dec3(torch.cat([u3, e3], dim=1))

        u2 = self.up2(d3)
        if u2.shape[2:] != e2.shape[2:]:
            u2 = self._center_crop_or_pad(u2, e2.shape[2:])
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.shape[2:] != e1.shape[2:]:
            u1 = self._center_crop_or_pad(u1, e1.shape[2:])
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        out = self.out_conv(d1)  # (B,3,D,H,W) raw logits
        return out

    @staticmethod
    def _center_crop_or_pad(x, target_spatial):
        _, _, D, H, W = x.shape
        tD, tH, tW = target_spatial
        dD = tD - D
        dH = tH - H
        dW = tW - W

        padD = (max(0, dD//2), max(0, dD - dD//2))
        padH = (max(0, dH//2), max(0, dH - dH//2))
        padW = (max(0, dW//2), max(0, dW - dW//2))
        if any(p > 0 for p in (*padD, *padH, *padW)):
            x = F.pad(x, (padW[0], padW[1], padH[0], padH[1], padD[0], padD[1]))

        _, _, D, H, W = x.shape
        sD = max(0, (D - tD)//2)
        sH = max(0, (H - tH)//2)
        sW = max(0, (W - tW)//2)
        x = x[:, :, sD:sD+tD, sH:sH+tH, sW:sW+tW]
        return x

model = UNet3D(in_channels=4, out_channels=3, base_ch=64).to(device)
model = model.to(memory_format=torch.channels_last_3d)
print("Model params:", sum(p.numel() for p in model.parameters())/1e6, "M")


Model params: 21.708163 M


# Loss (BCE+Dice) and MONAI metrics (Dice, HD95) + IoU

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # BCE
        bce_loss = self.bce(logits, targets)
        # Dice (multi-label across channels)
        probs = torch.sigmoid(logits)
        dims = (0, 2, 3, 4)  # sum over batch and spatial dims; keep channel dim
        intersection = (probs * targets).sum(dim=dims)
        denom = probs.sum(dim=dims) + targets.sum(dim=dims) + self.smooth
        dice = (2. * intersection + self.smooth) / denom
        dice_loss = 1. - dice.mean()
        return bce_loss + dice_loss

criterion = DiceBCELoss().to(device)

# Post transforms for predictions/labels
post_pred  = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(threshold=0.5)])

def binarize_with_thresholds(probs, thr=(0.35, 0.50, 0.50), enforce_union=True):
    """
    probs: (B,3,D,H,W) after sigmoid; channel order = [WT, TC, ET]
    thr: thresholds for WT, TC, ET respectively
    """
    assert probs.shape[1] == 3, "Expect 3 channels (WT,TC,ET)"
    b0 = (probs[:, 0] > thr[0])
    b1 = (probs[:, 1] > thr[1])
    b2 = (probs[:, 2] > thr[2])
    pred = torch.stack([b0, b1, b2], dim=1).float()
    if enforce_union:
        # WT should include TC and ET (keeps hierarchy consistent at eval)
        pred[:, 0] = (pred[:, 0].bool() | pred[:, 1].bool() | pred[:, 2].bool()).float()
    return pred

# ========= EVAL THRESHOLDS (WT, TC, ET) =========
import json, os
EVAL_THRESHOLDS = [0.35, 0.50, 0.50]  # initial guess; will be updated by calibrator
THR_PATH = os.path.join(CHECKPOINT_DIR, "eval_thresholds.json")

def save_eval_thresholds(thr, path=THR_PATH):
    with open(path, "w") as f:
        json.dump({"WT": thr[0], "TC": thr[1], "ET": thr[2]}, f)
    print(f"[Calibrator] Saved thresholds to {path}: (WT,TC,ET) = {thr}")

def load_eval_thresholds(path=THR_PATH):
    global EVAL_THRESHOLDS
    if os.path.exists(path):
        obj = json.load(open(path))
        EVAL_THRESHOLDS = [float(obj["WT"]), float(obj["TC"]), float(obj["ET"])]
        print(f"[Calibrator] Loaded thresholds from {path}: (WT,TC,ET) = {EVAL_THRESHOLDS}")
    else:
        print(f"[Calibrator] No thresholds file at {path}; using defaults {EVAL_THRESHOLDS}")

# ========= Calibrator (quick grid search) =========
from contextlib import nullcontext

def calibrate_thresholds(val_loader, model, device, candidates_w=(0.25,0.30,0.35,0.40),
                         candidates_t=(0.45,0.50), candidates_e=(0.45,0.50),
                         enforce_union=True, amp_enabled=True):
    """
    Returns best (WT,TC,ET) thresholds and the achieved mean Dice on the val set.
    """
    model.eval()
    best_thr = None
    best_score = -1.0

    # keep this lightweight: use DiceMetric once per combo, reset per batch
    for tw in candidates_w:
        for tt in candidates_t:
            for te in candidates_e:
                dice_ch_tmp = DiceMetric(include_background=False, reduction="mean_channel")
                with torch.inference_mode():
                    cmgr = torch.amp.autocast("cuda", dtype=autocast_dtype) if (amp_enabled and device=="cuda") else nullcontext()
                    for images, targets, _ in val_loader:
                        images = images.to(device).to(memory_format=torch.channels_last_3d)
                        targets = targets.to(device)
                        with cmgr:
                            logits = model(images)
                            probs = torch.sigmoid(logits)
                        preds = binarize_with_thresholds(probs, thr=(tw, tt, te), enforce_union=enforce_union)
                        dice_ch_tmp(y_pred=preds, y=targets)

                dch = torch.nan_to_num(dice_ch_tmp.aggregate(), nan=0.0).cpu()
                dice_ch_tmp.reset()
                # mean over WT,TC,ET
                mean_dice = float(dch.mean().item() if dch.numel() > 0 else 0.0)

                if mean_dice > best_score:
                    best_score = mean_dice
                    best_thr = (float(tw), float(tt), float(te))

    return best_thr, best_score


# Optimizer, (optional) Scheduler, TensorBoard writer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

writer = SummaryWriter(LOG_DIR)
print("TensorBoard log dir:", LOG_DIR)


TensorBoard log dir: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/tb_logs


# Checkpoint utilities (save/load, resume from any epoch)

In [ ]:
def ckpt_path_for_epoch(epoch: int) -> str:
    return os.path.join(CHECKPOINT_DIR, f"epoch_{epoch:03d}.pth")

def save_checkpoint(epoch, model, optimizer, scheduler, scaler=None, extra=None):
    path = ckpt_path_for_epoch(epoch)
    payload = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler else None,
        "scaler_state": scaler.state_dict() if scaler else None,
        "config": {
            "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "amp": AMP,
            "batch_size": BATCH_SIZE, "seed": SEED
        }
    }
    if extra:
        payload["extra"] = extra
    torch.save(payload, path)
    shutil.copy2(path, os.path.join(CHECKPOINT_DIR, "latest.pth"))
    return path

def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None, map_location=None):
    ckpt = torch.load(path, map_location=map_location or device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer and ckpt.get("optimizer_state") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler and ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler and ckpt.get("scaler_state") is not None:
        scaler.load_state_dict(ckpt["scaler_state"])
    print(f"Loaded checkpoint: {path} (epoch={ckpt.get('epoch', 'N/A')})")
    return ckpt.get("epoch", 0)


# Training & Validation loops (CSV + TensorBoard logging)

In [ ]:
from contextlib import nullcontext
from tqdm.notebook import tqdm

autocast_dtype = torch.bfloat16 if (device=="cuda") else None
scaler = torch.amp.GradScaler(device, enabled=(AMP and device=="cuda"))

# init CSV header once
if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, "w") as f:
        f.write("epoch,phase,loss,"
                "dice_mean,dice_wt,dice_tc,dice_et,"
                "hd95_mean,hd95_wt,hd95_tc,hd95_et,"
                "iou_mean,iou_wt,iou_tc,iou_et,lr\n")

def run_epoch(data_loader, phase: str, epoch: int):
    is_train = (phase == "train")
    model.train(is_train)
    torch.set_grad_enabled(is_train)

    running_loss = 0.0
    batch_count  = 0

    # ---- Epoch-level MONAI metrics ----
    # Always track Dice + IoU
    dice_mean_m = DiceMetric(include_background=False, reduction="mean")
    iou_mean_m  = MeanIoU(include_background=False, reduction="mean")
    dice_ch_m   = [DiceMetric(include_background=False, reduction="mean") for _ in range(3)]
    iou_ch_m    = [MeanIoU(include_background=False, reduction="mean") for _ in range(3)]

    # HD95 only for validation
    if phase == "val":
        hd95_mean_m = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
        hd95_ch_m   = [HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean") for _ in range(3)]
    else:
        hd95_mean_m, hd95_ch_m = None, [None, None, None]

    # Track if we ever updated a class this epoch (WT, TC, ET)
    ch_seen = [0, 0, 0]

    # tqdm
    batch_pos = 1 if is_train else 2
    batch_bar = tqdm(total=len(data_loader),
                     desc=f"{phase} {epoch:03d}",
                     position=batch_pos, leave=False, dynamic_ncols=True)

    cmgr = (lambda: torch.amp.autocast(device, dtype=autocast_dtype)) if (AMP and device=="cuda") else nullcontext

    for step, (images, targets, names) in enumerate(data_loader, 1):
        images  = images.to(device, non_blocking=True).to(memory_format=torch.channels_last_3d)  # (B,4,D,H,W)
        targets = targets.to(device, non_blocking=True)                                         # (B,3,D,H,W)

        with cmgr():
            logits = model(images)
            loss   = criterion(logits, targets)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if GRAD_CLIP_NORM > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()

        # ------- metrics on CPU; aggregate later -------
        with torch.inference_mode():
            if phase == "val":
                probs = torch.sigmoid(logits)
                preds = binarize_with_thresholds(probs, thr=tuple(EVAL_THRESHOLDS), enforce_union=True)
            else:
                preds = post_pred(logits)

            labs  = post_label(targets)

            # move to CPU to avoid GPU cache growth in MONAI accumulators
            p_cpu = preds.float().cpu()
            l_cpu = labs.float().cpu()

            # Whole-batch mean metrics (safe; NaNs unlikely when any class appears)
            dice_mean_m(y_pred=p_cpu, y=l_cpu)
            if phase == "val": hd95_mean_m(y_pred=p_cpu, y=l_cpu)
            iou_mean_m(y_pred=p_cpu,  y=l_cpu)

            # Per-class updates: do it per-case to gate by presence
            B = p_cpu.shape[0]
            for b in range(B):
                # (1,3,D,H,W)
                pb = p_cpu[b:b+1]
                lb = l_cpu[b:b+1]
                for c in range(3):  # 0:WT, 1:TC, 2:ET
                    gt_pos  = (lb[:, c].sum() > 0)
                    pr_pos  = (pb[:, c].sum() > 0)
                    if gt_pos or pr_pos:
                        # singleton channel tensors (shape: (1,1,D,H,W))
                        pbc = pb[:, c:c+1]
                        lbc = lb[:, c:c+1]
                        dice_ch_m[c](y_pred=pbc, y=lbc)
                        if phase == "val": hd95_ch_m[c](y_pred=pbc, y=lbc)
                        iou_ch_m[c](y_pred=pbc,  y=lbc)
                        ch_seen[c] += 1

        running_loss += loss.item()
        batch_count  += 1

        if step % max(1, PRINT_EVERY) == 0:
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")
            batch_bar.update(PRINT_EVERY)

        del logits, preds, labs, p_cpu, l_cpu

    # finish tqdm nicely
    remaining = len(data_loader) - batch_bar.n
    if remaining > 0: batch_bar.update(remaining)
    batch_bar.close()

    # ------- Aggregate ONCE per epoch -------
    epoch_loss = running_loss / max(1, batch_count)

    # Means
    d_mean = float(torch.nan_to_num(dice_mean_m.aggregate(), nan=0.0).item())
    if phase == "val":
        h_mean = float(torch.nan_to_num(hd95_mean_m.aggregate(), nan=0.0).item())
    else:
        h_mean = 0.0
    i_mean = float(torch.nan_to_num(iou_mean_m.aggregate(),  nan=0.0).item())

    # Per-class (WT, TC, ET): fill if never seen this epoch
    dice_vals = []
    hd95_vals = []
    iou_vals  = []
    for c in range(3):
        if ch_seen[c] > 0:
            dc = float(torch.nan_to_num(dice_ch_m[c].aggregate(), nan=0.0).item())
            if phase == "val":
                hc = float(torch.nan_to_num(hd95_ch_m[c].aggregate(), nan=0.0).item())
            else:
                hc = 0.0 # HD95 is not computed for training
            ic = float(torch.nan_to_num(iou_ch_m[c].aggregate(),  nan=0.0).item())
        else:
            # Class never present the entire epoch: perfect absence agreement
            dc, hc, ic = 1.0, 0.0, 1.0
        dice_vals.append(dc); hd95_vals.append(hc); iou_vals.append(ic)

    dice_wt, dice_tc, dice_et = dice_vals
    hd95_wt, hd95_tc, hd95_et = hd95_vals
    iou_wt,  iou_tc,  iou_et  = iou_vals

    # TensorBoard
    lr = optimizer.param_groups[0]["lr"]
    writer.add_scalar(f"{phase}/loss",      epoch_loss, epoch)
    writer.add_scalar(f"{phase}/dice_mean", d_mean,     epoch)
    writer.add_scalar(f"{phase}/hd95_mean", h_mean,     epoch)
    writer.add_scalar(f"{phase}/iou_mean",  i_mean,     epoch)

    # CSV (all numeric, no blanks, no zeros-from-NaNs)
    with open(METRICS_CSV, "a") as f:
        f.write(
            f"{epoch},{phase},{epoch_loss:.6f},"
            f"{d_mean:.6f},{dice_wt:.6f},{dice_tc:.6f},{dice_et:.6f},"
            f"{h_mean:.6f},{hd95_wt:.6f},{hd95_tc:.6f},{hd95_et:.6f},"
            f"{i_mean:.6f},{iou_wt:.6f},{iou_tc:.6f},{iou_et:.6f},"
            f"{lr:.8f}\n"
        )

    return epoch_loss, d_mean, h_mean, i_mean


def train_loop(num_epochs, resume_checkpoint="", resume_epoch=None):
    def prune_csv_from_epoch(csv_path, start_epoch):
        if not os.path.exists(csv_path):
            return
        df = pd.read_csv(csv_path)
        df = df[df["epoch"] < start_epoch]
        df.to_csv(csv_path, index=False)

    start_epoch = 1

    # resume logic
    if resume_checkpoint and os.path.exists(resume_checkpoint):
        start_epoch = load_checkpoint(resume_checkpoint, model, optimizer, scheduler, scaler) + 1
        prune_csv_from_epoch(METRICS_CSV, start_epoch)
    elif isinstance(resume_epoch, int):
        maybe = os.path.join(CHECKPOINT_DIR, f"epoch_{resume_epoch:03d}.pth")
        if os.path.exists(maybe):
            start_epoch = load_checkpoint(maybe, model, optimizer, scheduler, scaler) + 1
            prune_csv_from_epoch(METRICS_CSV, start_epoch)
        else:
            tqdm.write(f"Requested RESUME_EPOCH={resume_epoch} but file missing: {maybe}")


    # --- top-level Epochs bar pinned at position 0 ---
    epoch_bar = tqdm(range(start_epoch, num_epochs + 1),
                     desc="Epochs",
                     position=0,
                     leave=True,
                     dynamic_ncols=True)

    best_val_dice = -1.0
    for epoch in epoch_bar:
        t0 = time.time()
        train_loss, train_dice, train_hd95, train_iou = run_epoch(train_loader, "train", epoch)
        val_loss,   val_dice,   val_hd95,   val_iou   = run_epoch(val_loader,   "val",   epoch)

        # step scheduler on val loss
        scheduler.step(val_loss)

        # save
        if SAVE_EVERY_EPOCH:
            path = save_checkpoint(epoch, model, optimizer, scheduler, scaler)
            tqdm.write(f"[Epoch {epoch:03d}] Saved checkpoint: {path}")

        # track best
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            shutil.copy2(ckpt_path_for_epoch(epoch), os.path.join(CHECKPOINT_DIR, "best_by_val_dice.pth"))

        dt = time.time() - t0
        # update the epoch bar line (no extra prints)
        epoch_bar.set_postfix(
            train_loss=f"{train_loss:.4f}",
            val_loss=f"{val_loss:.4f}",
            val_dice=f"{val_dice:.4f}",
            val_iou=f"{val_iou:.4f}",
            sec=f"{dt:.1f}",
        )

        # inside train_loop, right after you compute val_* for the epoch:
        if epoch == 25:
            print(f"[Calibrator] Running threshold search at epoch {epoch}...")
            thr, score = calibrate_thresholds(val_loader, model, device, amp_enabled=AMP)
            if thr is not None:
                global EVAL_THRESHOLDS
                EVAL_THRESHOLDS = list(thr)
                save_eval_thresholds(EVAL_THRESHOLDS)  # write to disk for test-time reuse
                print(f"[Calibrator] New thresholds (WT,TC,ET) = {EVAL_THRESHOLDS}, val mean Dice = {score:.4f}")

        # force a render refresh in some Colab skins
        epoch_bar.refresh()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

# Run training

In [ ]:
# Kick off training. To resume from a checkpoint, set RESUME_CHECKPOINT or RESUME_EPOCH in the config cell.
train_loop(NUM_EPOCHS, resume_checkpoint=RESUME_CHECKPOINT, resume_epoch=RESUME_EPOCH)
print("Training complete.")


Loaded checkpoint: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/epoch_042.pth (epoch=42)


Epochs:   0%|          | 0/8 [00:00<?, ?it/s]

train 043:   0%|          | 0/438 [00:00<?, ?it/s]

val 043:   0%|          | 0/94 [00:00<?, ?it/s]

monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
the ground truth of class 1 is all 0, this may result in nan/inf distance.
the ground truth of class 0 is all 0, this may result in nan/inf distance.


...

val 049:   0%|          | 0/94 [00:00<?, ?it/s]

[Epoch 049] Saved checkpoint: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/epoch_049.pth


train 050:   0%|          | 0/438 [00:00<?, ?it/s]

val 050:   0%|          | 0/94 [00:00<?, ?it/s]

[Epoch 050] Saved checkpoint: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/epoch_050.pth
Training complete.


# Load final/best checkpoint and evaluate on test set

In [ ]:
# =========================
# Evaluate on test set (WT, TC, ET) with robust per-subject metrics
# =========================
EVAL_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "best_by_val_dice.pth")  # or "epoch_050.pth"
_ = load_checkpoint(EVAL_CHECKPOINT, model, optimizer=None, scheduler=None, scaler=None)

model.eval().to(device, memory_format=torch.channels_last_3d)

from contextlib import nullcontext
cmgr = (lambda: torch.amp.autocast("cuda", dtype=torch.bfloat16)) if (AMP and device=="cuda") else nullcontext

# --- Per-class accumulators across the whole test set (WT, TC, ET) ---
dice_c = [DiceMetric(include_background=False, reduction="mean") for _ in range(3)]
iou_c  = [MeanIoU(include_background=False,   reduction="mean") for _ in range(3)]
hd95_c = [HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean") for _ in range(3)]
seen_c = [0, 0, 0]  # track if class was ever present (gt or pred) across the test set

per_subj_rows = []

per_subj_rows = []

with torch.inference_mode():
    for images, targets, names in test_loader:
        images  = images.to(device, non_blocking=True).to(memory_format=torch.channels_last_3d)
        targets = targets.to(device, non_blocking=True)

        with cmgr():
            logits = model(images)

        preds = post_pred(logits)    # (1,3,D,H,W) in {0,1}
        labs  = post_label(targets)  # (1,3,D,H,W) in {0,1}

        # ---------- update per-class test-set accumulators ----------
        ps = preds.float().cpu()
        ls = labs.float().cpu()

        for c in range(3):  # 0:WT, 1:TC, 2:ET
            p_c = ps[:, c:c+1]
            l_c = ls[:, c:c+1]
            has_pred = (p_c.sum() > 0)
            has_gt   = (l_c.sum() > 0)
            if has_pred or has_gt:
                dice_c[c](y_pred=p_c, y=l_c)
                iou_c[c](y_pred=p_c,  y=l_c)
                hd95_c[c](y_pred=p_c, y=l_c)
                seen_c[c] += 1

        # ---------- robust per-subject numbers for table ----------
        per_cls_dice, per_cls_iou, per_cls_hd95 = [], [], []
        for c in range(3):
            p_c = ps[:, c:c+1]
            l_c = ls[:, c:c+1]
            has_pred = (p_c.sum() > 0)
            has_gt   = (l_c.sum() > 0)

            if not has_pred and not has_gt:
                # perfect absence agreement for this subject
                per_cls_dice.append(1.0); per_cls_iou.append(1.0); per_cls_hd95.append(0.0)
                continue

            d_m = DiceMetric(include_background=False, reduction="mean")
            i_m = MeanIoU(include_background=False,   reduction="mean")
            h_m = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
            d_m(y_pred=p_c, y=l_c); i_m(y_pred=p_c, y=l_c); h_m(y_pred=p_c, y=l_c)

            d_val = float(torch.nan_to_num(d_m.aggregate(), nan=0.0).item())
            i_val = float(torch.nan_to_num(i_m.aggregate(), nan=0.0).item())
            h_val = float(torch.nan_to_num(h_m.aggregate(), nan=0.0).item())

            per_cls_dice.append(d_val); per_cls_iou.append(i_val); per_cls_hd95.append(h_val)

        per_subj_rows.append({
            "subject": names[0],
            "dice_wt": per_cls_dice[0], "dice_tc": per_cls_dice[1], "dice_et": per_cls_dice[2],
            "iou_wt":  per_cls_iou[0],  "iou_tc":  per_cls_iou[1],  "iou_et":  per_cls_iou[2],
            "hd95_wt": per_cls_hd95[0], "hd95_tc": per_cls_hd95[1], "hd95_et": per_cls_hd95[2],
        })

# === Aggregate per-class across the whole test set (guaranteed exactly 3 values) ===
def agg_or_fill(metric_obj, seen, fill):
    if seen > 0:
        return float(torch.nan_to_num(metric_obj.aggregate(), nan=0.0).item())
    else:
        return fill

dice_wt = agg_or_fill(dice_c[0], seen_c[0], 1.0)
dice_tc = agg_or_fill(dice_c[1], seen_c[1], 1.0)
dice_et = agg_or_fill(dice_c[2], seen_c[2], 1.0)

iou_wt  = agg_or_fill(iou_c[0],  seen_c[0], 1.0)
iou_tc  = agg_or_fill(iou_c[1],  seen_c[1], 1.0)
iou_et  = agg_or_fill(iou_c[2],  seen_c[2], 1.0)

hd95_wt = agg_or_fill(hd95_c[0], seen_c[0], 0.0)
hd95_tc = agg_or_fill(hd95_c[1], seen_c[1], 0.0)
hd95_et = agg_or_fill(hd95_c[2], seen_c[2], 0.0)

dice_mean = float((dice_wt + dice_tc + dice_et) / 3.0)
iou_mean  = float((iou_wt  + iou_tc  + iou_et ) / 3.0)
hd95_mean = float((hd95_wt + hd95_tc + hd95_et) / 3.0)

# === Save ===
summary_path = os.path.join(CHECKPOINT_DIR, "test_metrics.csv")
pd.DataFrame([{
    "dice_mean": dice_mean, "dice_wt": dice_wt, "dice_tc": dice_tc, "dice_et": dice_et,
    "iou_mean":  iou_mean,  "iou_wt":  iou_wt,  "iou_tc":  iou_tc,  "iou_et":  iou_et,
    "hd95_mean": hd95_mean, "hd95_wt": hd95_wt, "hd95_tc": hd95_tc, "hd95_et": hd95_et,
}]).to_csv(summary_path, index=False)

per_case_path = os.path.join(CHECKPOINT_DIR, "test_metrics_per_subject.csv")
pd.DataFrame(per_subj_rows).to_csv(per_case_path, index=False)

print("=== Test Summary ===")
print(f"Dice (WT/TC/ET): {dice_wt:.4f} / {dice_tc:.4f} / {dice_et:.4f} | mean={dice_mean:.4f}")
print(f"IoU  (WT/TC/ET): {iou_wt:.4f} / {iou_tc:.4f} / {iou_et:.4f} | mean={iou_mean:.4f}")
print(f"HD95 (WT/TC/ET): {hd95_wt:.2f} / {hd95_tc:.2f} / {hd95_et:.2f} | mean={hd95_mean:.2f}")
print("Saved:", summary_path)
print("Saved:", per_case_path)

Loaded checkpoint: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/best_by_val_dice.pth (epoch=49)
=== Test Summary ===
Dice (WT/TC/ET): 0.9246 / 0.8931 / 0.8703 | mean=0.8960
IoU  (WT/TC/ET): 0.8712 / 0.8425 / 0.7999 | mean=0.8379
HD95 (WT/TC/ET): 4.71 / 3.93 / 2.93 | mean=3.85
Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/test_metrics.csv
Saved: /content/drive/MyDrive/Brain_Tumor_Segmentation/DHW_baseline_checkpoints/test_metrics_per_subject.csv


# Visualization — overlay predictions vs labels on T2-FLAIR (axial slices)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

VIS_DIR = os.path.join(CHECKPOINT_DIR, "visualizations")
os.makedirs(VIS_DIR, exist_ok=True)

def overlay_and_save(image_4ch, pred_3ch, true_3ch, out_path, title=""):
    """
    image_4ch: (4,D,H,W) — we use T2-FLAIR (index 2) for background
    pred_3ch, true_3ch: (3,D,H,W) in {0,1}; channel order: WT, TC, ET
    Saves a single axial slice near the middle of the volume.
    """
    D = image_4ch.shape[1]
    z = D // 2

    t2f = image_4ch[2, z].cpu().numpy()  # (H,W)

    p_wt = pred_3ch[0, z].cpu().numpy().astype(bool)
    p_tc = pred_3ch[1, z].cpu().numpy().astype(bool)
    p_et = pred_3ch[2, z].cpu().numpy().astype(bool)

    y_wt = true_3ch[0, z].cpu().numpy().astype(bool)
    y_tc = true_3ch[1, z].cpu().numpy().astype(bool)
    y_et = true_3ch[2, z].cpu().numpy().astype(bool)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(t2f, cmap="gray")
    axes[0].imshow(np.ma.masked_where(~y_wt, y_wt), alpha=0.4)
    axes[0].imshow(np.ma.masked_where(~y_tc, y_tc), alpha=0.4)
    axes[0].imshow(np.ma.masked_where(~y_et, y_et), alpha=0.4)
    axes[0].set_title("Ground Truth (WT/TC/ET)")
    axes[0].axis("off")

    axes[1].imshow(t2f, cmap="gray")
    axes[1].imshow(np.ma.masked_where(~p_wt, p_wt), alpha=0.4)
    axes[1].imshow(np.ma.masked_where(~p_tc, p_tc), alpha=0.4)
    axes[1].imshow(np.ma.masked_where(~p_et, p_et), alpha=0.4)
    axes[1].set_title("Prediction (WT/TC/ET)")
    axes[1].axis("off")

    axes[2].imshow(t2f, cmap="gray")
    axes[2].imshow(np.ma.masked_where(~(p_wt ^ y_wt), (p_wt ^ y_wt)), alpha=0.4)
    axes[2].imshow(np.ma.masked_where(~(p_tc ^ y_tc), (p_tc ^ y_tc)), alpha=0.4)
    axes[2].imshow(np.ma.masked_where(~(p_et ^ y_et), (p_et ^ y_et)), alpha=0.4)
    axes[2].set_title("Error map (XOR)")
    axes[2].axis("off")

    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

# Export a few cases from test set
model.eval()
exported = 0
with torch.no_grad():
    for images, targets, names in test_loader:
        images = images.to(device)
        targets = targets.to(device)
        logits = model(images)
        preds = post_pred(logits)
        for i in range(images.shape[0]):
            out_path = os.path.join(VIS_DIR, f"{names[i]}_overlay.png")
            overlay_and_save(images[i].cpu(), preds[i].cpu(), targets[i].cpu(), out_path, title=names[i])
            exported += 1
        if exported >= 5:
            break
print(f"Saved {exported} overlay visualizations to:", VIS_DIR)


# Plot learning curves (Loss, Dice, IoU, HD95) from CSV

In [ ]:
df = pd.read_csv(METRICS_CSV)
fig_dir = os.path.join(CHECKPOINT_DIR, "figures")
os.makedirs(fig_dir, exist_ok=True)

def plot_metric(metric):
    plt.figure()
    for phase in ["train", "val"]:
        sub = df[df["phase"] == phase]
        plt.plot(sub["epoch"], sub[metric], label=phase)
    plt.xlabel("Epoch")
    plt.ylabel(metric.upper())
    plt.title(metric.upper())
    plt.legend()
    outp = os.path.join(fig_dir, f"{metric}_curve.png")
    plt.savefig(outp, bbox_inches="tight")
    plt.close()
    print("Saved:", outp)

for m in ["loss", "dice", "iou", "hd95"]:
    plot_metric(m)


# Launch TensorBoard (Colab)

In [ ]:
# In Colab, run:
# %load_ext tensorboard
# %tensorboard --logdir $LOG_DIR
print("To view TensorBoard in Colab:\n  %load_ext tensorboard\n  %tensorboard --logdir $LOG_DIR")
